# Pydantic model to structure output


TODO: skapa en agent som ska simulera en IT-anställd OR
create an agent to simulate IT employees

In [ ]:
from pydantic_ai import Agent
from dotenv import load_dotenv
load_dotenv()

employee_simulator_agent = Agent("openrouter:liquid/lfm-2.5-1.2b-thinking:free",
system_prompt=""""
You are an HR expert within IT field in Swden within data science, data engineering, machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- name
- age
- gender
- job_title
- salary in SEK per month
""")

result = await employee_simulator_agent.run("Simulate two employees")


In [ ]:
print(result.output)

In [ ]:
with open("simulated _employees.md", "w") as file:
    file.write(result.output)

## Get more strutured output

issue with above:
- output structure vary
- hard to work with the data e.g compute mean of salaries

want:

- repeatable structure

In [32]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent

class EmployeeModel(BaseModel):
    name: str
    age: int = Field(description="age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        gte=30_000, lte=50_000, description="salary should be between 30k and 50k, the higher experience level, the higher salary"
    )

employee_simulator_agent = Agent("openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering, machine learning, AI engineering. You will simulate IT employees.   
""",
)

result = await employee_simulator_agent.run(user_prompt="Give me 3 employees", output_type=list[EmployeeModel])

C:\Users\tasmi\AppData\Local\Temp\ipykernel_332\3490425939.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(


In [ ]:
result.output


In [ ]:
result.output[1]

In [ ]:
result = await employee_simulator_agent.run("Give me 3 employees", output_type=list[EmployeeModel])
result

In [ ]:
result.output[0]


In [ ]:
#BaseModel -> dictionary
result.output[0].model_dump()

TODO:
- result.output make into list of dictionaries
- create pandas dataframe based on this list 
- export a csv file of our simulated employees

In [ ]:
import pandas as pd

# list_of_dicts = [item.__dict__ for item in result.output]

list_of_dicts = [employee.model_dump() for employee in result.output]

list_of_dicts


df = pd.DataFrame(list_of_dicts)

print(df.head())

In [21]:
employees = result.output
employees

[EmployeeModel(name='Emma Johnson', age=35, gender='Female', experience_level='Senior', job_title='Machine Learning Engineer', salary=42000),
 EmployeeModel(name='Liam Carter', age=42, gender='Male', experience_level='Expert', job_title='Data Scientist', salary=48000),
 EmployeeModel(name='Ava Rodriguez', age=22, gender='Female', experience_level='Entry', job_title='Data Engineer', salary=32000)]

In [22]:
employees_list = [employee.model_dump() for employee in employees]
employees_list



[{'name': 'Emma Johnson',
  'age': 35,
  'gender': 'Female',
  'experience_level': 'Senior',
  'job_title': 'Machine Learning Engineer',
  'salary': 42000},
 {'name': 'Liam Carter',
  'age': 42,
  'gender': 'Male',
  'experience_level': 'Expert',
  'job_title': 'Data Scientist',
  'salary': 48000},
 {'name': 'Ava Rodriguez',
  'age': 22,
  'gender': 'Female',
  'experience_level': 'Entry',
  'job_title': 'Data Engineer',
  'salary': 32000}]

In [24]:
import pandas as pd
df = pd.DataFrame(employees_list)
df

,name,age,gender,experience_level,job_title,salary
0,Emma Johnson,35,Female,Senior,Machine Learning Engineer,42000
1,Liam Carter,42,Male,Expert,Data Scientist,48000
2,Ava Rodriguez,22,Female,Entry,Data Engineer,32000


In [25]:
df["salary"].mean()


np.float64(40666.666666666664)

In [28]:
df.to_csv("simulated_employees.csv", index=False)